[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/beltrami_fields/beltrami_fields.ipynb)

# Beltrami Fields

This example computes force-free Beltrami magnetic fields inside a conducting toroidal cavity and follows a tangled field as resistive damping relaxes it toward the lowest-energy state of the same magnetic linkage. In a Beltrami field, electric current flows along the magnetic field itself, so the magnetic force on the current vanishes everywhere.

In spacetime, an observer at rest with the cavity has worldline tangent `time`, and a static magnetic field is modelled as a spatial bivector whose flux through an oriented surface patch reads as `field & patch`. The term ***simplex gradient*** here means the piecewise-constant 1-form `(face_patches & time) / (3 * prisms)` that sweeps each tetrahedron's outward face patch along `time` over three times its swept spacetime volume, building the edge Whitney 1-forms and their curls `gradients[tails] ^ gradients[heads]` without any spatial metric. Magnetic energy pulls the vacuum duality map `Field.dual()` back through `curl.adjugate()`, whereas magnetic helicity wedges the potential 1-form with signed volume `Plane ^ volumes` and pulls that metric-free pairing back to the same edge circulations; single-tetrahedron Whitney forms are developed in [Whitney Forms](../../surfaces/whitney_forms/whitney_forms.ipynb).

In Gibbs-Heaviside vector notation, $\mathbf B = \nabla\times\mathbf A$ is treated as an axial vector field satisfying $\nabla\times\mathbf B = \lambda\mathbf B$ at stationary points of $\frac{1}{2}\int|\mathbf B|^2\,dV - \frac{\lambda}{2}\int\mathbf A\cdot(\nabla\times\mathbf A)\,dV$. Discretizing those integrals with tetrahedral Nédélec edge elements $\mathbf w_{ij} = \lambda_i\nabla\lambda_j - \lambda_j\nabla\lambda_i$ expresses $\nabla\lambda_i = J^{-T}\hat\nabla\hat\lambda_i$ through a $3\times 3$ reference-tetrahedron Jacobian inverse $J^{-T}$ and evaluates both stiffness $S_{ee'} = \int (\nabla\times\mathbf w_e)\cdot(\nabla\times\mathbf w_{e'})\,|\det J|\,d\hat V$ and helicity $H_{ee'} = \int \mathbf w_e\cdot(\nabla\times\mathbf w_{e'})\,|\det J|\,d\hat V$ with Euclidean dot and cross products.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import spdiag
from numga.topology import Topology
from examples.electromagnetism.beltrami_fields import render
from examples.electromagnetism.beltrami_fields.numerical import coexact_helicity_eigh
from examples.electromagnetism.beltrami_fields.plumbing import flux_preserving, solid_torus
from examples.util.animation import save_figure, show_states

ctx = NumpyContext(ga)
mv = ctx.multivector
Scalar = ga.gatype.scalar()                                 # Scalar+
Plane = ga.gatype.vector()                                  # Vector+
SpatialPlane = Plane.derive.within("x y z")                 # Vector+
# Static magnetic fields lie in the spatial bivector planes of the cavity rest frame.
Field = ga.gatype.bivector().derive.within("x y z")         # Bivector+
Point = ga.gatype.antivector()                              # Antivector-
Volume = ga.gatype.antitrivector()                          # Antitrivector-
Antiscalar = ga.gatype.antiscalar()                         # Antiscalar-
OneForm = ga.gatype((SpatialPlane, Scalar))                 # Plane+ <- Scalar+
Curl = ga.gatype((Field, Scalar))                           # Field+ <- Scalar+
Form = ga.gatype((Antiscalar, Scalar))                      # Antiscalar- <- Scalar+

# Worldline tangent of an observer at rest with the torus, pairing to one with `mv.t`.
time = -mv.t.dual()                                         # [] Point-

## Beltrami modes in a torus

Magnetic energy reads the field through the vacuum duality map, while magnetic helicity reads its linkage through signed volume. Both pull back to symmetric forms on edge circulations, and a Beltrami mode makes their responses proportional. Reflecting a field across a plane preserves its energy on positively oriented tetrahedra while reversing its helicity, so the modes come in opposite-handedness pairs.

In [ ]:
major_radius = 1.0
minor_radius = 0.42
n_major = 16
n_minor = 8
shown_indices = np.array([-1, -4])                                              # [2]
n_shown = len(shown_indices)

In [ ]:
def simplex_volumes(topology: Topology, vertices: Point) -> Volume:
    """The three edges from each tetrahedron's first corner span six times its signed volume (`Volume-[T]`)."""
    edges = topology.oriented[3, 1] * (topology.d[0] * vertices)                # [6] Point-[T]
    return (edges[0] & edges[1] & edges[2]) / 6                                 # Volume-[T]


def simplex_gradients(topology: Topology, vertices: Point, volumes: Volume) -> SpatialPlane:
    """Sweeping each outward face patch along `time` gives the four corner simplex gradients (`[4] Plane+[T]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                # [3] Point-[F]
    face_patches = topology.boundary[3, 2] * ((sides[0] & sides[1]) / 2)        # [4] Patch-[T]
    prisms = volumes & time                                                     # Antiscalar-[T]
    return (face_patches & time) / (3 * prisms)                                 # [4] Plane+[T]


topology, vertices = solid_torus(major_radius, minor_radius, n_major, n_minor)  # Topology, Point-[V]
centres = topology.incidence[3, 0].mean(axis=0) * vertices                      # Point-[T]
volumes = simplex_volumes(topology, vertices)                                   # Volume-[T]
gradients = simplex_gradients(topology, vertices, volumes)                      # [4] Plane+[T]

In [ ]:
def whitney_1form(topology: Topology, gradients: SpatialPlane) -> OneForm:
    """Each edge's Whitney 1-form evaluates to one quarter of its endpoint gradient difference at centroids (`(Plane+ <- Scalar+)[T <- E]`)."""
    # At each tetrahedron centroid the four corner weights equal 1/4, reducing the edge 1-form
    # to one quarter of the difference of its endpoints' simplex gradients.
    tails, heads = topology.corners[3, 1]
    centroid_1forms = (gradients[heads] - gradients[tails]) / 4                 # [6] Plane+[T]
    return (spdiag(centroid_1forms * Scalar) * topology.oriented[3, 1]).sum(axis=0)  # (Plane+ <- Scalar+)[T <- E]


def whitney_curl(topology: Topology, gradients: SpatialPlane) -> Curl:
    """The exterior derivative of the edge Whitney 1-forms gives the magnetic field map (`(Field+ <- Scalar+)[T <- E]`)."""
    # Each edge's Whitney curl is twice the wedge of its endpoints' simplex gradients, assembled
    # with the relative edge orientations `topology.oriented[3, 1]`.
    tails, heads = topology.corners[3, 1]
    curls = 2 * (gradients[tails] ^ gradients[heads])                           # [6] Field+[T]
    return (spdiag(curls * Scalar) * topology.oriented[3, 1]).sum(axis=0)       # (Field+ <- Scalar+)[T <- E]


potential_map = whitney_1form(topology, gradients)                              # (Plane+ <- Scalar+)[T <- E]
curl = whitney_curl(topology, gradients)                                        # (Field+ <- Scalar+)[T <- E]

In [ ]:
def magnetic_energy(volumes: Volume, curl: Curl) -> Form:
    """Pulling the volume-weighted vacuum duality map back through `curl` gives the magnetic energy form (`(Antiscalar- <- Scalar+)[E <- E]`)."""
    vacuum = (volumes & time).norm() * Field.dual()                             # (Excitation- <- Field+)[T]
    return curl.adjugate()(vacuum(curl))                                        # (Antiscalar- <- Scalar+)[E <- E]


def magnetic_helicity(volumes: Volume, potential_map: OneForm, curl: Curl) -> Form:
    """Wedging `potential_map` with `volumes` and pulling back through `curl` gives the symmetrized helicity form (`(Antiscalar- <- Scalar+)[E <- E]`)."""
    volume_weights = Plane ^ volumes                                            # (Excitation- <- Plane+)[T]
    raw_helicity = curl.adjugate()(volume_weights(potential_map))               # (Antiscalar- <- Scalar+)[E <- E]
    return (raw_helicity + raw_helicity.adjugate()) / 2                         # (Antiscalar- <- Scalar+)[E <- E]


stiffness = magnetic_energy(volumes, curl)                                      # (Antiscalar- <- Scalar+)[E <- E]
helicity = magnetic_helicity(volumes, potential_map, curl)                      # (Antiscalar- <- Scalar+)[E <- E]

A perfectly conducting cavity wall blocks outward magnetic flux across every boundary triangle, while one closed poloidal circulation around the tube remains free to carry net toroidal magnetic flux.

In [ ]:
free = flux_preserving(topology, n_minor)                                       # Scalar+[E <- E_free]
# Zero-energy edge gradients change the potential without changing the magnetic field.
free_energy = free.adjugate() * stiffness * free                                # (Antiscalar- <- Scalar+)[E_free <- E_free]
free_helicity = free.adjugate() * helicity * free                               # (Antiscalar- <- Scalar+)[E_free <- E_free]
inverse_eigenvalues, free_modes, n_null = coexact_helicity_eigh(free_energy, free_helicity)
modes = free * free_modes                                                       # [...] Scalar+[E]
shown_modes = modes[shown_indices]                                              # [n_shown] Scalar+[E]
shown_fields = curl(shown_modes)                                                # [n_shown] Field+[T]

save_figure(
    render.eigenmodes_figure(topology, vertices, centres, volumes, inverse_eigenvalues, shown_fields, shown_indices),
    "beltrami_eigenmodes",
);

On the left, the spectrum of inverse Beltrami eigenvalues pairs positive (vermilion) and negative (sapphire) helicities symmetrically across zero. Highlighted markers single out the fundamental positive-helicity Taylor state in the middle panel and an excited mode with finer internal twist in the right panel, with field-line tubes coloured from vermilion on the inner flux shell through teal to sapphire on the outer shell and warmed toward gold where lines climb steeply.

In finite-element exterior calculus notation, the generalized eigenvalue problem reads $S a = \lambda H a$, with stiffness matrix $S_{ee'} = \int dw_e \wedge \star dw_{e'}$ and helicity matrix $H_{ee'} = \frac{1}{2}\int (w_e \wedge dw_{e'} + w_{e'} \wedge dw_e)$.

## Selective decay

Resistive diffusion damps fine-scale twists faster than broad circulations while magnetic reconnection nearly conserves total magnetic helicity. Rescaling selectively damped Beltrami modes at constant total helicity models Woltjer-Taylor relaxation from a tangled superposition toward the lowest-energy mode carrying that linkage.

Tracing and rendering the 3D field-line tubes across the animation takes about ten seconds.

In [ ]:
selected_indices = np.array([-1, -2, -4, -8, 3])                                # [5]
n_selected = len(selected_indices)
initial_weights = np.array([0.45, 0.70, 0.85, 0.85, 0.45])                      # [n_selected]
n_frames = 46
relaxation_time = 0.025
decay_times = np.linspace(0.0, relaxation_time, n_frames)                       # [n_frames]
frame_duration_ms = 40

In [ ]:
def relaxation_states(
    initial_amplitudes: Scalar,
    eigenvalues: Scalar,
    modes: Scalar,
    curl: Curl,
    times: np.ndarray,
) -> Iterator[tuple[Scalar, Field]]:
    """Yields modal amplitudes and reconstructed magnetic fields (`[...] Scalar+`, `Field+[T]`) under helicity-preserving decay."""
    initial_helicity = (initial_amplitudes.squared() / eigenvalues).sum(axis=0)  # [] Scalar+
    for decay_time in times:
        # Finer modes decay faster; the shared scale fixes the linkage of the whole field.
        raw_amplitudes = initial_amplitudes * (-eigenvalues.squared() * decay_time).exp()  # [...] Scalar+
        raw_helicity = (raw_amplitudes.squared() / eigenvalues).sum(axis=0)                # [] Scalar+
        amplitudes = raw_amplitudes * (initial_helicity / raw_helicity).square_root()      # [...] Scalar+
        cochain = (amplitudes * modes).sum(axis=0)                                         # Scalar+[E]
        yield amplitudes, curl(cochain)


selected_modes = modes[selected_indices]                                        # [n_selected] Scalar+[E]
selected_eigenvalues = 1 / inverse_eigenvalues[selected_indices]                # [n_selected] Scalar+
initial_amplitudes = mv.scalar(initial_weights[:, None])                        # [n_selected] Scalar+
streamed_states = show_states(
    relaxation_states(initial_amplitudes, selected_eigenvalues, selected_modes, curl, decay_times),
    render.relaxation_animation,
    topology,
    vertices,
    centres,
    volumes,
    name="beltrami_relaxation",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
)

In the oblique and axial views, tangled multi-mode windings smooth out as higher-eigenvalue ripples decay, leaving the nested helical flux surfaces of the fundamental Taylor state at the minimum magnetic energy compatible with the conserved helicity.

In spectral magnetohydrodynamic notation, expanding the field in orthonormal Chandrasekhar-Kendall modes gives $E = \frac{1}{2}\sum_k c_k^2$ and $H = \sum_k c_k^2 / \lambda_k$, so damping $c_k(t) \propto c_k(0)e^{-\lambda_k^2 t}$ at fixed $H$ drives $E$ monotonically toward the Woltjer-Taylor bound $\frac{1}{2}|\lambda_1 H|$.

In [ ]:
# checks
linear_potential = (mv.x & vertices) * 0.6 - (mv.y & vertices) * 0.9 + (mv.z & vertices) * 0.4
exact_gradient = (gradients * (topology.incidence[3, 0] * linear_potential)).sum(axis=0)
assert ((potential_map(topology.d[0] * linear_potential) - exact_gradient).norm() < 1e-11).all()
assert (curl(topology.d[0] * linear_potential).norm() < 1e-11).all()
assert ((topology.reading(topology.exterior[2], 2) * (topology.d[1] * shown_modes[0])).norm() < 1e-12).all()

assert n_null == n_major
assert ((inverse_eigenvalues + inverse_eigenvalues[::-1]).norm() < 1e-11).all()
shown_eigenvalues = 1 / inverse_eigenvalues[shown_indices]
shown_energies = (stiffness(shown_modes) & shown_modes).sites.sum() / 2
shown_helicities = (helicity(shown_modes) & shown_modes).sites.sum()
shown_residuals = free.adjugate() * (stiffness(shown_modes) - shown_eigenvalues * helicity(shown_modes))
assert ((shown_energies - 0.5).norm() < 1e-11).all()
assert ((shown_helicities - inverse_eigenvalues[shown_indices]).norm() < 1e-11).all()
assert (shown_residuals.dual_inverse().norm().sites.sum() < 1e-10).all()

amplitude_frames = stack([amplitudes for amplitudes, _ in streamed_states])
energy_frames = amplitude_frames.squared().sum(axis=1) / 2
helicity_frames = (amplitude_frames.squared() / selected_eigenvalues).sum(axis=1)
taylor_bound = helicity_frames[0] * selected_eigenvalues[0] / 2
assert ((helicity_frames - helicity_frames[0]).norm() < 1e-11).all()
assert (energy_frames[1:] < energy_frames[:-1]).all()
assert energy_frames[-1] > taylor_bound
assert (energy_frames[-1] - taylor_bound) / taylor_bound < 0.01